# Day 1: Multi-Agent Incident Triage & Orchestration

## Advanced Multi-Agent AI Systems with LangChain & LangGraph
### For Support & Meta Engineers

---

## 🎯 What You'll Learn

In this hands-on lab, you will:
1. Build multi-agent systems using **LangChain** and **LangGraph**
2. Orchestrate complex workflows with **StateGraph**
3. Implement structured outputs with **Pydantic schemas**
4. Add production **guardrails** (max steps, loop detection, tool allowlists)
5. Enable **observability** with JSONL trace logging
6. Handle **failure modes** and recovery patterns

## 📊 What You'll Build

A **3-agent incident triage system** that:
- **Classifies** incidents (severity P0-P4, category)
- **Deduplicates** similar incidents (alert storm detection)
- **Routes** to appropriate teams (SLA-aware)

All orchestrated via **LangGraph StateGraph** with full observability.

---

## Part 1: Setup & Environment

### 1.1 Import Dependencies

In [ ]:
# Install project dependencies (safe to re-run)
# Optimized for Day 1 - removed unnecessary packages
%pip install -q \
    langchain>=0.1.0 \
    langchain-openai>=0.0.5 \
    langgraph>=0.0.20 \
    langchain-core>=0.1.0 \
    openai>=1.12.0 \
    python-dotenv>=1.0.0 \
    pydantic>=2.0.0 \
    pandas>=2.0.0 \
    numpy>=1.24.0 \
    scikit-learn>=1.3.0 \
    typing-extensions>=4.5.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [53]:
import sys
import os
import json
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✓ Project root: {project_root}")
print(f"✓ Python version: {sys.version.split()[0]}")

✓ Project root: c:\Users\bharg\Downloads\Work\advanced-multi-agent-ai-systems-2-half-days
✓ Python version: 3.13.5


### 1.2 Initialize LLM (MockLLM or OpenAI)

**IMPORTANT**: This notebook runs **without API keys** using MockLLM.

To use OpenAI:
1. Create `.env` file in project root
2. Add: `OPENAI_API_KEY=sk-your-key-here`
3. Restart kernel

In [54]:
from src.llm_factory import get_llm, print_llm_stats

llm = get_llm(force_mock=False, deterministic=True, verbose=True)

print("\n✓ LLM initialized and ready")

🚀 Running with ChatOpenAI
   Model: gpt-4o-mini

✓ LLM initialized and ready


### 1.3 Load Sample Data

In [30]:
incidents = []
with open(project_root / 'data' / 'incidents_large_small.jsonl', 'r') as f:
    for line in f:
        incidents.append(json.loads(line.strip()))

with open(project_root / 'data' / 'kb_policies.json', 'r') as f:
    policies = json.load(f)

with open(project_root / 'data' / 'sample_logs.txt', 'r') as f:
    sample_logs = f.read()

print(f"✓ Loaded {len(incidents)} incidents")
print(f"✓ Loaded policies: {list(policies.keys())}")
print(f"✓ Loaded {len(sample_logs.splitlines())} log lines")

print("\nSample Incidents:")
for i, inc in enumerate(incidents[:3], 1):
    print(f"{i}. {inc['id']}: {inc['title']} [{inc['severity']}]")

✓ Loaded 20 incidents
✓ Loaded policies: ['sla', 'routing', 'escalation', 'automation_rules', 'compliance', 'notification_templates']
✓ Loaded 49 log lines

Sample Incidents:
1. INC-10001: Database Connection Pool Exhausted [P0]
2. INC-10002: API Gateway Returning 503 Errors [P1]
3. INC-10003: Memory Leak in Payment Service [P1]


---

## Part 2: Pydantic Schemas for Structured Outputs

### 2.1 Understanding Structured Outputs

**Why Pydantic schemas?**
- ✅ Type safety and validation
- ✅ Automatic error detection
- ✅ Clear contracts between agents
- ✅ Production-ready data models

In [31]:
from src.schemas import (
    ClassificationResult,
    DeduplicationResult,
    RoutingDecision,
    SeverityLevel,
    IncidentCategory
)

print("Available Schemas:")
print(f"  - ClassificationResult: {ClassificationResult.__fields__.keys()}")
print(f"  - DeduplicationResult: {DeduplicationResult.__fields__.keys()}")
print(f"  - RoutingDecision: {RoutingDecision.__fields__.keys()}")

print("\nSeverity Levels:", [s.value for s in SeverityLevel])
print("Categories:", [c.value for c in IncidentCategory])

Available Schemas:
  - ClassificationResult: dict_keys(['severity', 'category', 'confidence', 'reasoning', 'requires_escalation'])
  - DeduplicationResult: dict_keys(['is_duplicate', 'duplicate_of', 'similar_incidents', 'confidence', 'recommendation'])
  - RoutingDecision: dict_keys(['assigned_team', 'sla_hours', 'priority', 'escalation_required', 'reasoning', 'confidence'])

Severity Levels: ['P0', 'P1', 'P2', 'P3', 'P4']
Categories: ['Database', 'Network', 'Application', 'Infrastructure', 'Security', 'Unknown']


C:\Users\bharg\AppData\Local\Temp\ipykernel_196180\2850540034.py:10: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"  - ClassificationResult: {ClassificationResult.__fields__.keys()}")
C:\Users\bharg\AppData\Local\Temp\ipykernel_196180\2850540034.py:11: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"  - DeduplicationResult: {DeduplicationResult.__fields__.keys()}")
C:\Users\bharg\AppData\Local\Temp\ipykernel_196180\2850540034.py:12: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 t

### 2.2 Schema Validation Example

In [32]:
try:
    valid_classification = ClassificationResult(
        severity=SeverityLevel.P0,
        category=IncidentCategory.DATABASE,
        confidence=0.95,
        reasoning="Database connection pool exhausted",
        requires_escalation=True
    )
    print("✓ Valid classification created")
    print(json.dumps(valid_classification.dict(), indent=2))
except Exception as e:
    print(f"✗ Validation error: {e}")

print("\n" + "="*60)
print("Testing Invalid Data:")
print("="*60)

try:
    invalid_classification = ClassificationResult(
        severity="P99",
        category="InvalidCategory",
        confidence=1.5,
        reasoning="Test"
    )
except Exception as e:
    print(f"✓ Caught validation error (expected): {type(e).__name__}")
    print(f"  Message: {str(e)[:100]}...")

✓ Valid classification created
{
  "severity": "P0",
  "category": "Database",
  "confidence": 0.95,
  "reasoning": "Database connection pool exhausted",
  "requires_escalation": true
}

Testing Invalid Data:
✓ Caught validation error (expected): ValidationError
  Message: 3 validation errors for ClassificationResult
severity
  Input should be 'P0', 'P1', 'P2', 'P3' or 'P...


C:\Users\bharg\AppData\Local\Temp\ipykernel_196180\2114450045.py:10: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(json.dumps(valid_classification.dict(), indent=2))


---

## Part 3: LangGraph StateGraph Workflow

### 3.1 Understanding StateGraph

**LangGraph StateGraph** is a state machine for orchestrating agents:
- **Nodes**: Individual agent operations
- **Edges**: Transitions between nodes
- **State**: Shared data passed between nodes
- **Conditional edges**: Dynamic routing based on state

```
┌─────────────┐
│  Classify   │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Guardrails  │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Deduplicate │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│    Route    │
└─────────────┘
```

### 3.2 Initialize Observability

In [33]:
from src.observability import TraceLogger, create_trace_viewer

logger = TraceLogger(log_file="traces_day1.jsonl")
logger.clear()

print("✓ Trace logger initialized")
print(f"  Log file: traces_day1.jsonl")

✓ Trace logger initialized
  Log file: traces_day1.jsonl


### 3.3 Create Triage Workflow

In [34]:
from src.workflow_day1 import create_triage_workflow

triage_workflow = create_triage_workflow(
    llm=llm,
    logger=logger,
    max_steps=10,
    incident_db=incidents,
    policies=policies
)

print("✓ Triage workflow created")
print(f"  Max steps: 10")
print(f"  Incident DB size: {len(incidents)}")

✓ Triage workflow created
  Max steps: 10
  Incident DB size: 20


### 3.4 Run Single Incident Through Workflow

In [35]:
test_incident = incidents[0]

print("\n" + "="*60)
print(f"Processing: {test_incident['id']}")
print(f"Title: {test_incident['title']}")
print(f"Description: {test_incident['description'][:100]}...")
print("="*60 + "\n")

result = triage_workflow.run(test_incident)

print("\n" + "="*60)
print("Workflow Result:")
print("="*60)
print(f"Status: {result['status']}")
print(f"Trace ID: {result['trace_id']}")
print(f"Steps: {result['step_count']}")
print(f"Errors: {len(result.get('errors', []))}")


Processing: INC-10001
Title: Database Connection Pool Exhausted
Description: Production database experiencing connection pool exhaustion. Users unable to access application. Err...


Workflow Result:
Status: completed
Trace ID: 195c1515-dc13-4d07-992d-732bcbb01873
Steps: 3
Errors: 0


### 3.5 Examine Classification Result

In [36]:
classification = result.get('classification', {})

print("\n📊 CLASSIFICATION RESULT")
print("="*60)
if classification:
    print(json.dumps(classification, indent=2))
else:
    print("No classification result available")


📊 CLASSIFICATION RESULT
{
  "severity": "P1",
  "category": "Database",
  "confidence": 0.9,
  "reasoning": "The incident involves a production database that is critical for application access, affecting a significant number of users (1500). The connection pool exhaustion indicates a severe issue that needs immediate attention but does not necessarily halt all operations.",
  "requires_escalation": true
}


### 3.6 Examine Deduplication Result

In [37]:
deduplication = result.get('deduplication', {})

print("\n📊 DEDUPLICATION RESULT")
print("="*60)
if deduplication:
    print(json.dumps(deduplication, indent=2))
else:
    print("No deduplication result available")


📊 DEDUPLICATION RESULT
{
  "is_duplicate": false,
  "duplicate_of": null,
  "similar_incidents": [],
  "confidence": 0.8,
  "recommendation": "Create new incident"
}


### 3.7 Examine Routing Result

In [38]:
routing = result.get('routing', {})

print("\n📊 ROUTING RESULT")
print("="*60)
if routing:
    print(json.dumps(routing, indent=2))
else:
    print("No routing result available")


📊 ROUTING RESULT
{
  "assigned_team": "Database Team",
  "sla_hours": 4,
  "priority": 1,
  "escalation_required": false,
  "reasoning": "The incident involves a critical database issue that requires immediate attention from the Database Team, which specializes in handling database-related incidents.",
  "confidence": 0.85
}


---

## Part 4: Observability & Trace Viewing

### 4.1 View Trace Summary

In [39]:
trace_summary = logger.get_trace_summary(result['trace_id'])

print("\n📊 TRACE SUMMARY")
print("="*60)
print(json.dumps(trace_summary, indent=2))


📊 TRACE SUMMARY
{
  "trace_id": "195c1515-dc13-4d07-992d-732bcbb01873",
  "workflow_name": "TriageWorkflow",
  "status": "completed",
  "total_duration_ms": 4259.4170570373535,
  "node_timings": {
    "classify": 1543.879,
    "deduplicate": 883.178,
    "route": 1825.267
  },
  "tool_calls_count": 0,
  "errors_count": 0,
  "violations_count": 0,
  "total_events": 8
}


### 4.2 View All Traces as DataFrame

In [40]:
traces_df = create_trace_viewer(logger)

print("\n📊 ALL TRACES")
print("="*60)
if not traces_df.empty:
    display(traces_df)
else:
    print("No traces available")


📊 ALL TRACES


,trace_id,workflow_name,status,total_duration_ms,node_timings,tool_calls_count,errors_count,violations_count,total_events
0,195c1515-dc13-4d07-992d-732bcbb01873,TriageWorkflow,completed,4259.417057,"{'classify': 1543.879, 'deduplicate': 883.178,...",0,0,0,8


### 4.3 Detailed Trace Visualization

In [41]:
from src.observability import format_trace_for_display

trace_display = format_trace_for_display(logger, result['trace_id'])
print(trace_display)

Trace ID: 195c1515-dc13-4d07-992d-732bcbb01873

🚀 Workflow Started: TriageWorkflow
   Time: 2026-01-27T16:21:13.730084

▶️  Node: classify
   ✓ Completed in 1543.88ms

▶️  Node: deduplicate
   ✓ Completed in 883.18ms

▶️  Node: route
   ✓ Completed in 1825.27ms

✅ Workflow Ended: completed
   Time: 2026-01-27T16:21:17.989509


---

## Part 5: Guardrails in Action

### 5.1 Max Steps Guardrail

Test what happens when max steps is exceeded:

In [42]:
print("\n" + "="*60)
print("Testing Max Steps Guardrail")
print("="*60 + "\n")

limited_workflow = create_triage_workflow(
    llm=llm,
    logger=logger,
    max_steps=2,
    incident_db=incidents,
    policies=policies
)

result_limited = limited_workflow.run(incidents[1])

print(f"Status: {result_limited['status']}")
print(f"Steps executed: {result_limited['step_count']}")
print(f"Errors: {result_limited.get('errors', [])}")


Testing Max Steps Guardrail

Status: completed
Steps executed: 3
Errors: []


### 5.2 Error Handling

Test handling of invalid JSON responses:

In [43]:
print("\n" + "="*60)
print("Testing Error Handling")
print("="*60 + "\n")

invalid_incident = {
    "id": "TEST-001",
    "title": "",
    "description": "",
    "severity": "unknown"
}

result_error = triage_workflow.run(invalid_incident)

print(f"Status: {result_error['status']}")
print(f"Errors encountered: {len(result_error.get('errors', []))}")
if result_error.get('errors'):
    print("\nError details:")
    for i, error in enumerate(result_error['errors'][:3], 1):
        print(f"  {i}. {error[:100]}...")


Testing Error Handling

Status: completed_with_errors
Errors encountered: 1

Error details:
  1. Invalid JSON from deduplication: Expecting value: line 1 column 1 (char 0)...


---

## Part 6: Batch Processing

### 6.1 Process Multiple Incidents

In [44]:
print("\n" + "="*60)
print("Batch Processing Incidents")
print("="*60 + "\n")

batch_results = []

for i, incident in enumerate(incidents[:5], 1):
    print(f"{i}. Processing {incident['id']}: {incident['title'][:40]}...")
    
    result = triage_workflow.run(incident)
    
    batch_results.append({
        'incident_id': incident['id'],
        'status': result['status'],
        'step_count': result['step_count'],
        'has_errors': len(result.get('errors', [])) > 0,
        'classification': result.get('classification', {}).get('severity'),
        'routing': result.get('routing', {}).get('assigned_team')
    })
    
    print(f"   ✓ {result['status']} - {result['step_count']} steps\n")

print("\n" + "="*60)
print("Batch Summary")
print("="*60)

batch_df = pd.DataFrame(batch_results)
display(batch_df)


Batch Processing Incidents

1. Processing INC-10001: Database Connection Pool Exhausted...
   ✓ completed - 3 steps

2. Processing INC-10002: API Gateway Returning 503 Errors...
   ✓ completed - 3 steps

3. Processing INC-10003: Memory Leak in Payment Service...
   ✓ completed - 3 steps

4. Processing INC-10004: Disk Space Critical on Log Server...
   ✓ completed - 3 steps

5. Processing INC-10005: Authentication Service Slow Response...
   ✓ completed - 3 steps


Batch Summary


,incident_id,status,step_count,has_errors,classification,routing
0,INC-10001,completed,3,False,SeverityLevel.P1,Database Team
1,INC-10002,completed,3,False,SeverityLevel.P1,Infrastructure Team
2,INC-10003,completed,3,False,SeverityLevel.P1,Application Support Team
3,INC-10004,completed,3,False,SeverityLevel.P1,Infrastructure Team
4,INC-10005,completed,3,False,SeverityLevel.P1,Application Support Team


### 6.2 Analyze Batch Results

In [45]:
print("\n📊 BATCH ANALYSIS")
print("="*60)

success_rate = (batch_df['status'] == 'completed').sum() / len(batch_df)
avg_steps = batch_df['step_count'].mean()
error_rate = batch_df['has_errors'].sum() / len(batch_df)

print(f"Total Incidents: {len(batch_df)}")
print(f"Success Rate: {success_rate:.1%}")
print(f"Average Steps: {avg_steps:.1f}")
print(f"Error Rate: {error_rate:.1%}")

print("\nClassification Distribution:")
print(batch_df['classification'].value_counts())

print("\nRouting Distribution:")
print(batch_df['routing'].value_counts())


📊 BATCH ANALYSIS
Total Incidents: 5
Success Rate: 100.0%
Average Steps: 3.0
Error Rate: 0.0%

Classification Distribution:
classification
SeverityLevel.P1    5
Name: count, dtype: int64

Routing Distribution:
routing
Infrastructure Team         2
Application Support Team    2
Database Team               1
Name: count, dtype: int64


---

## Part 7: Alert Storm Detection

### 7.1 Detect Duplicate Incidents

In [46]:
from src.clustering import detect_alert_storm, IncidentClusterer

print("\n" + "="*60)
print("Alert Storm Detection")
print("="*60 + "\n")

alert_storm_result = detect_alert_storm(
    incidents=incidents[:15],
    time_window_minutes=60,
    min_incidents=3
)

print(json.dumps(alert_storm_result, indent=2))


Alert Storm Detection

{
  "is_alert_storm": false,
  "incident_count": 15,
  "clusters_found": 0,
  "message": "No significant clusters detected"
}


### 7.2 Cluster Similar Incidents

In [47]:
clusterer = IncidentClusterer(similarity_threshold=0.70)
clusters = clusterer.cluster_incidents(incidents[:10], min_cluster_size=2)

print("\n📊 INCIDENT CLUSTERS")
print("="*60)
print(f"Total clusters found: {len(clusters)}\n")

for cluster_id, cluster_incidents in clusters.items():
    print(f"\n{cluster_id}:")
    print(f"  Size: {len(cluster_incidents)}")
    
    summary = clusterer.get_cluster_summary(cluster_incidents)
    print(f"  Common Severity: {summary['most_common_severity']}")
    print(f"  Common Category: {summary['most_common_category']}")
    print(f"  Incidents: {summary['incident_ids']}")


📊 INCIDENT CLUSTERS
Total clusters found: 0



---

## Part 8: LangChain Tools

### 8.1 Understanding Tools

**LangChain Tools** provide structured interfaces for:
- **Read operations**: Search, query, lookup (safe)
- **Write operations**: Update, escalate (require approval)

**Tool Allowlists** are a key guardrail for production.

In [48]:
from src.tools import get_read_only_tools, get_default_tools

read_only_tools = get_read_only_tools(
    incident_db=incidents,
    policies=policies,
    log_data=sample_logs
)

print("\n📦 READ-ONLY TOOLS")
print("="*60)
for tool in read_only_tools:
    print(f"  - {tool.name}: {tool.description[:60]}...")

all_tools = get_default_tools(
    incident_db=incidents,
    policies=policies,
    log_data=sample_logs,
    allow_writes=True
)

print("\n📦 ALL TOOLS (including writes)")
print("="*60)
for tool in all_tools:
    print(f"  - {tool.name}: {tool.description[:60]}...")


📦 READ-ONLY TOOLS
  - search_incidents: Search for similar incidents in the knowledge base. Use this...
  - lookup_policy: Look up organizational policies for SLA, escalation, and rou...
  - query_logs: Query logs for a specific service and time range. Read-only ...
  - query_metrics: Query metrics for a specific service. Read-only operation....

📦 ALL TOOLS (including writes)
  - search_incidents: Search for similar incidents in the knowledge base. Use this...
  - lookup_policy: Look up organizational policies for SLA, escalation, and rou...
  - query_logs: Query logs for a specific service and time range. Read-only ...
  - query_metrics: Query metrics for a specific service. Read-only operation....
  - escalate_incident: Escalate an incident to a higher-level team. REQUIRES APPROV...
  - update_incident: Update incident fields. REQUIRES APPROVAL for critical chang...


### 8.2 Test Tool Execution

In [49]:
search_tool = read_only_tools[0]

print("\n🔧 Testing Tool: search_incidents")
print("="*60)

result = search_tool.run({"query": "database", "limit": 3})
print(json.dumps(json.loads(result), indent=2))


🔧 Testing Tool: search_incidents
{
  "results": [
    {
      "incident_id": "INC-10001",
      "title": "Database Connection Pool Exhausted",
      "severity": "P0",
      "similarity_score": 0.85
    }
  ],
  "count": 1
}


---

## Part 9: Checkpoint & Review

### What We've Learned

✅ **LangGraph StateGraph** for workflow orchestration  
✅ **Pydantic schemas** for structured outputs  
✅ **Guardrails**: max steps, error handling  
✅ **Observability**: JSONL traces, trace IDs  
✅ **LangChain Tools** with read/write separation  
✅ **Alert storm detection** and clustering  

### LLM Usage Statistics

In [50]:
print_llm_stats(llm)


📊 LLM Usage Statistics
Model: ChatOpenAI
Note: Use LangSmith or callbacks for detailed tracking



### Trace Statistics

In [51]:
all_traces = create_trace_viewer(logger)

if not all_traces.empty:
    print("\n📊 TRACE STATISTICS")
    print("="*60)
    print(f"Total Traces: {len(all_traces)}")
    print(f"Completed: {(all_traces['status'] == 'completed').sum()}")
    print(f"Failed: {(all_traces['status'] == 'failed').sum()}")
    print(f"\nAverage Duration: {all_traces['total_duration_ms'].mean():.2f}ms")
    print(f"Total Tool Calls: {all_traces['tool_calls_count'].sum()}")
    print(f"Total Errors: {all_traces['errors_count'].sum()}")
    print(f"Total Violations: {all_traces['violations_count'].sum()}")
else:
    print("No trace data available")


📊 TRACE STATISTICS
Total Traces: 8
Completed: 8
Failed: 0

Average Duration: 4558.19ms
Total Tool Calls: 0
Total Errors: 1
Total Violations: 0


---

## 🎉 Day 1 Complete!

You've successfully built a **production-ready incident triage system** using:
- ✅ LangChain & LangGraph
- ✅ Structured outputs with Pydantic
- ✅ Production guardrails
- ✅ Full observability

### Next Steps

**Day 2** will cover:
- 5-agent root cause analysis system
- Advanced guardrails (loop detection, tool allowlists)
- Evaluation framework with metrics
- Failure modes and recovery
- Production deployment patterns

### Homework (Optional)

1. Add a 4th agent for incident summarization
2. Implement custom tool for querying metrics
3. Add conditional routing based on severity
4. Experiment with different similarity thresholds

See you tomorrow! 🚀